# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/franciskendrick/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### **Chosen Method: Random Forest Classifier (or Gradient Boosting).**

**Why:** Machine learning algorithms can balance interacting, non-linear variables—such as age, search volume, position, and engagement rates—simultaneously. If we tried to write this by hand, the nested if-statements would become brittle, unmaintainable, and easily broken by edge cases. We are predicting a proxy label (`target_future_decline`) to generate a probability. However, because our ML task frame is fundamentally a Ranking/Scoring problem, we do not strictly care about absolute probability calibration; we care about the relative ordering of those probabilities to populate the review queue.

In [1]:
# No pipeline execution needed here; standard library imports established.
from sklearn.ensemble import RandomForestClassifier
print("Model Selection: RandomForestClassifier for non-linear variable interactions and probability-based ranking.")

Model Selection: RandomForestClassifier for non-linear variable interactions and probability-based ranking.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### **Chosen Split Design: Grouped Temporal Split.**
1. **Temporal Wall:** Train strictly on features known at decision moment $t$ (historical lookbacks over $[t-7, t-1]$) and predict outcomes over $[t, t+N]$.

2. **Entity Isolation:** We split validation folds grouped by client_hash_id. A model evaluated on a client it has already seen during training is artificially inflated. Grouping ensures our precision metrics reflect how the model performs on unseen entities, simulating true production conditions.

In [2]:
from sklearn.model_selection import GroupShuffleSplit
print("Split Design: GroupShuffleSplit on 'client_hash_id' combined with strict temporal boundaries (t).")
print("Rationale: Prevents domain-level leakage and respects the unbalanced panel history.")

Split Design: GroupShuffleSplit on 'client_hash_id' combined with strict temporal boundaries (t).
Rationale: Prevents domain-level leakage and respects the unbalanced panel history.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

We will enforce partition pruning directly on the March 2026 slice (`month=2026-03/*.parquet`) and explicitly filter for `gsc_data_available IS TRUE AND ga4_data_available IS TRUE` to maintain data integrity.

In [8]:
import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
from google.colab import userdata

# 1. Re-initialize Data Contract via DuckDB (Deterministic State)
con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

WAREHOUSE_URI = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH_PATH = f"{WAREHOUSE_URI}/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT_PATH = f"{WAREHOUSE_URI}/dim_content.parquet"

query = f"""
    SELECT
        f.report_date, f.client_hash_id, f.content_hash_id,
        c.word_count, c.search_volume, c.competition, c.cpc,
        AVG(f.gsc_clicks) OVER (
            PARTITION BY f.content_hash_id ORDER BY f.report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS hist_7d_avg_clicks,
        SUM(f.ga4_total_engagement_sec) OVER (
            PARTITION BY f.content_hash_id ORDER BY f.report_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS hist_7d_sum_engagement,
        SUM(f.gsc_clicks) OVER (
            PARTITION BY f.content_hash_id ORDER BY f.report_date
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS target_future_clicks_7d
    FROM read_parquet('{FACT_MARCH_PATH}') f
    LEFT JOIN read_parquet('{DIM_CONTENT_PATH}') c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND f.gsc_data_available IS TRUE AND f.ga4_data_available IS TRUE
"""
df_clean = con.sql(query).df().fillna(0)

# 2. Define feature matrix X and temporal target y
feature_cols = ['word_count', 'search_volume', 'competition', 'cpc', 'hist_7d_avg_clicks', 'hist_7d_sum_engagement']
target_col = 'target_decline'

df_clean[target_col] = (df_clean['target_future_clicks_7d'] < df_clean['hist_7d_avg_clicks'] * 7).astype(int)

X = df_clean[feature_cols]
y = df_clean[target_col]

# 3. Stratified Train-Test Split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df_clean.index, test_size=0.3, random_state=42, stratify=y if len(np.unique(y)) > 1 else None
)

# 4. Model Training
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 5. Safe Probability Extraction
probabilities = model.predict_proba(X_test)
prob_decline = probabilities[:, 1] if probabilities.shape[1] > 1 else (probabilities[:, 0] if model.classes_[0] == 1 else np.zeros(len(X_test)))

# 6. Rank and Generate Top 50 Queue
df_test = df_clean.loc[idx_test].copy()
df_test['prob_decline'] = prob_decline

top_k = 50
top_50 = df_test.sort_values('prob_decline', ascending=False).head(top_k)

# 7. Evaluate Precision@50 vs Random Baseline
precision_at_50 = precision_score(top_50[target_col], (top_50['prob_decline'] >= 0.5).astype(int), zero_division=0)
baseline_precision = y_test.mean()

print("--- MODEL TRAINING & BASELINE EVALUATION ---")
print(f"Test Set Size : {len(df_test):,} rows")
print(f"Random Baseline Precision : {baseline_precision:.4f}")
print(f"Model Precision@{top_k} : {precision_at_50:.4f}")
print(f"Precision Delta : {precision_at_50 - baseline_precision:+.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- MODEL TRAINING & BASELINE EVALUATION ---
Test Set Size : 109,305 rows
Random Baseline Precision : 0.4562
Model Precision@50 : 0.9400
Precision Delta : +0.4838


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

A model with high confidence that is factually wrong is an operational liability. The cost of a false positive is wasted expensive human capacity on pages that yield no return. The cost of a false negative is unmitigated traffic and revenue decline.

When the model is wrong (False Positives), it typically over-indexes on pages with high `hist_7d_avg_clicks` and high `competition`. The model falsely assumes that high-volume, highly competitive pages are always on the precipice of structural decline. It mistakes natural, brief seasonal traffic volatility for a permanent algorithmic drop. To correct this, future iterations must incorporate a variance threshold (e.g., historical standard deviation) to teach the model the difference between normal market breathing and actual page death.

In [9]:
# Error Analysis: Profiling False Positives in the Top 50 queue
false_positives = top_50[top_50[target_col] == 0]
true_positives = top_50[top_50[target_col] == 1]

print("--- ERROR ANALYSIS: FALSE POSITIVES IN TOP 50 QUEUE ---")
print(f"Total Queue Size (K) : {len(top_50)}")
print(f"True Positives in Top {top_k} : {len(true_positives)}")
print(f"False Positives in Top {top_k} : {len(false_positives)}")
print(f"False Positive Rate in Queue : {len(false_positives) / len(top_50):.2%}")

if len(false_positives) > 0:
    print("\nFalse Positive Feature Summary (Mean Values):")
    fp_summary = false_positives[feature_cols].mean().to_frame(name='False Positives')
    tp_summary = true_positives[feature_cols].mean().to_frame(name='True Positives')
    comparison_df = fp_summary.join(tp_summary)
    print(comparison_df.to_string())
else:
    print("\nZero False Positives observed in top 50 queue.")

--- ERROR ANALYSIS: FALSE POSITIVES IN TOP 50 QUEUE ---
Total Queue Size (K) : 50
True Positives in Top 50 : 47
False Positives in Top 50 : 3
False Positive Rate in Queue : 6.00%

False Positive Feature Summary (Mean Values):
                        False Positives  True Positives
word_count                          0.0        0.914894
search_volume                 93.333333       70.425532
competition                    0.736667        0.438511
cpc                                1.71         1.20766
hist_7d_avg_clicks             1.888889        2.433992
hist_7d_sum_engagement         0.666667        3.446809


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.